In [ ]:
import os
import json
import numpy as np
import pandas as pd
import streamlit as st
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

# ----------------------------------------------------------------------
# 1. SETUP & CONFIGURATION
# ----------------------------------------------------------------------
st.set_page_config(page_title="SmartSwipe: Active Learning Declutter", layout="wide")
IMAGE_DIR = "photos"  # Create a folder named 'photos' with your images
DECISIONS_FILE = "swipe_decisions.json"
EXPORT_DIR = "delete_candidates"

if not os.path.exists(EXPORT_DIR):
    os.makedirs(EXPORT_DIR)

# Load device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------------------------------------------------------
# 2. FEATURE EXTRACTION ENGINE (PyTorch MobileNetV2)
# ----------------------------------------------------------------------
@st.cache_resource
def load_feature_extractor():
    # Load pre-trained MobileNetV2, freeze weights, strip classifier
    model = models.mobilenet_v2(pretrained=True)
    model.classifier = torch.nn.Identity()
    model.eval()
    return model.to(device)

def get_image_embedding(image_path, model):
    preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    try:
        img = Image.open(image_path).convert("RGB")
        tensor = preprocess(img).unsqueeze(0).to(device)
        with torch.no_grad():
            embedding = model(tensor).squeeze().cpu().numpy()
        return embedding / np.linalg.norm(embedding)  # L2 Normalize
    except Exception:
        return None

# ----------------------------------------------------------------------
# 3. ML MODEL FROM SCRATCH (NumPy Logistic Regression)
# ----------------------------------------------------------------------
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -15, 15)))

def train_classifier(X, y, lr=0.1, epochs=500):
    num_features = X.shape[1]
    weights = np.zeros(num_features)
    bias = 0.0

    for _ in range(epochs):
        z = np.dot(X, weights) + bias
        predictions = sigmoid(z)
        # Gradient computation
        dw = (1 / len(y)) * np.dot(X.T, (predictions - y))
        db = (1 / len(y)) * np.sum(predictions - y)
        # Updates
        weights -= lr * dw
        bias -= lr * db
    return weights, bias

# ----------------------------------------------------------------------
# 4. ACTIVE LEARNING ENGINE (Uncertainty Sampling)
# ----------------------------------------------------------------------
def initialize_session(image_files, extractor):
    if 'embeddings' not in st.session_state:
        st.session_state.embeddings = {}
        for f in image_files:
            path = os.path.join(IMAGE_DIR, f)
            emb = get_image_embedding(path, extractor)
            if emb is not None:
                st.session_state.embeddings[f] = emb

    if 'decisions' not in st.session_state:
        if os.path.exists(DECISIONS_FILE):
            with open(DECISIONS_FILE, 'r') as file:
                st.session_state.decisions = json.load(file)
        else:
            st.session_state.decisions = {}

if not os.path.exists(IMAGE_DIR) or len(os.listdir(IMAGE_DIR)) == 0:
    st.error(f"Please create a '{IMAGE_DIR}' folder and populate it with photos.")
    st.stop()

image_files = [f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
extractor = load_feature_extractor()
initialize_session(image_files, extractor)

# ----------------------------------------------------------------------
# 5. RETRAIN & QUEUE PREDICTION
# ----------------------------------------------------------------------
labeled_imgs = list(st.session_state.decisions.keys())
unlabeled_imgs = [f for f in st.session_state.embeddings.keys() if f not in labeled_imgs]

weights, bias = None, 0.0
uncertainty_df = []

if len(labeled_imgs) >= 4:  # Require minimum seed labels to activate AL loop
    X_train = np.array([st.session_state.embeddings[f] for f in labeled_imgs])
    y_train = np.array([st.session_state.decisions[f] for f in labeled_imgs])

    if len(np.unique(y_train)) > 1: # Require both classes (0: Keep, 1: Delete)
        weights, bias = train_classifier(X_train, y_train)

        # Predict uncertainties for unlabeled points
        for f in unlabeled_imgs:
            emb = st.session_state.embeddings[f]
            prob = sigmoid(np.dot(emb, weights) + bias)
            # Entropy/Uncertainty metric: closest to 0.5 probability is most uncertain
            uncertainty = 1.0 - abs(prob - 0.5) * 2
            uncertainty_df.append({'filename': f, 'prob_delete': prob, 'uncertainty': uncertainty})

        uncertainty_df = pd.DataFrame(uncertainty_df).sort_values(by='uncertainty', ascending=False)

# Select Next Image to Swipe
if len(uncertainty_df) > 0:
    next_image = uncertainty_df.iloc[0]['filename']  # Active Learning pick
else:
    next_image = unlabeled_imgs[0] if len(unlabeled_imgs) > 0 else None

# ----------------------------------------------------------------------
# 6. STREAMLIT INTERACTIVE UI
# ----------------------------------------------------------------------
st.title("📱 SmartSwipe: Active Learning Photo Declutter")
st.markdown(f"**Progress:** {len(labeled_imgs)} swiped / {len(st.session_state.embeddings)} total images")

col1, col2 = st.columns([2, 1])

with col1:
    if next_image:
        st.image(os.path.join(IMAGE_DIR, next_image), use_container_width=True, caption=f"Reviewing: {next_image}")

        # User Interface Swiping Framework
        btn_col1, btn_col2 = st.columns(2)
        with btn_col1:
            if st.button("🟢 KEEP (Swipe Right)", use_container_width=True):
                st.session_state.decisions[next_image] = 0
                with open(DECISIONS_FILE, 'w') as f:
                    json.dump(st.session_state.decisions, f)
                st.rerun()
        with btn_col2:
            if st.button("🔴 DELETE (Swipe Left)", use_container_width=True):
                st.session_state.decisions[next_image] = 1
                with open(DECISIONS_FILE, 'w') as f:
                    json.dump(st.session_state.decisions, f)
                st.rerun()
    else:
        st.success("🎉 All photos processed! Check your metrics and export folder below.")

with col2:
    st.header("⚙️ Active Learning Loop Backend")
    if weights is not None:
        st.metric(label="Model Status", value="Active & Optimizing")
        st.write("Top Uncertain Candidates Queue:")
        st.dataframe(uncertainty_df[['filename', 'uncertainty']].head(5), hide_index=True)
    else:
        st.info("⚠️ Swipe at least 2 KEEPs and 2 DELETEs to initialize active uncertainty sampling loop.")

# Confidence slider and batch export pipeline
st.divider()
st.header("📦 Automation and Local Export")
threshold = st.slider("Confidence Threshold for Auto-Deletion Strategy", 0.0, 1.0, 0.7)

if st.button("Automate Remaining & Export Delete Candidates"):
    if weights is not None and len(unlabeled_imgs) > 0:
        auto_deleted = 0
        for f in unlabeled_imgs:
            emb = st.session_state.embeddings[f]
            prob = sigmoid(np.dot(emb, weights) + bias)
            if prob >= threshold:
                # Symlink or save to target delete folder
                img_src = os.path.join(IMAGE_DIR, f)
                img_dest = os.path.join(EXPORT_DIR, f)
                Image.open(img_src).save(img_dest)
                auto_deleted += 1
        st.success(f"Successfully processed remaining pool! Automated export of **{auto_deleted} delete candidates** to `/{EXPORT_DIR}` folder based on model convergence.")
    else:
        st.warning("Please build model training history by manual swiping before executing full automation.")